In [1]:
import os
from typing import Optional, Dict, List
import json

import pandas as pd
import numpy as np
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = '1'
os.environ["MKL_NUM_THREADS"] = '1'
os.environ["OPENBLAS_NUM_THREADS"] = '1'
os.environ["VECLIB_MAXIMUM_THREADS"] = '1'
os.environ["NUMEXPR_NUM_THREADS"] = '1'

In [3]:
tf_adata = io.read_tfad(os.path.join(data_path, 'processed', author + '_consensus_tf_activity.h5ad'))

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def ood_split(tf_adata,
               train_frac: float = 0.8,
               min_pert_frac: float = 0.4,
               min_cat_frac: float = 0.5,
               deviation_thresh: float = 0.025,
               max_attempts: int = 1000,
               exclude_pert_control: bool = True,
               pert_col: str = 'ligand',
               ctrl_pert: str ='CTRL',
               seed: int = 888 
                   ):
    """
    Create a train-test split for the McCauley dataset. 
    Splits by condition (`pert_col` + `cat_col`) and corresponding barcodes 
    Also ensures a minimum fraction of total of each perturbation / category in conditions is in the train split.

    Parameters
    ----------
    tf_adata : _type_
        _description_
    train_frac : float, optional
        fraction of conditions to split into train, by default 0.8
    min_pert_frac : float, optional
        Minimum per-perturbation training coverage, defined at the level of
        conditions. For each perturbation p, at least `min_pert_frac` of all
        conditions associated with p must be included in the training set:

            (# training conditions with p) /
            (# total conditions with p) >= min_pert_frac

        This constraint enforces sufficient coverage of each perturbation in
        training, rather than merely ensuring that a perturbation is seen at
        least once (more than zero-shot).

        **Toy example**

        If perturbation A appears in 10 conditions total and `min_pert_frac = 0.4`,
        then at least 4 of those conditions must be in the training set. A split
        with only 1 training condition for A (1/10) would be rejected, even
        though A is technically present in training.

    min_cat_frac : float, optional
        same as `min_pert_frac`, but for category rather than perturbation

    exclude_pert_control : bool, optional
        whether to ensure all control perturbations are in the train (True) or not (False), by default True
    max_attempts : int, optional
        maximum number of tries to achieve a split that meets these requirements, by default 1000
    seed : int, optional
        random state, by default 888
    """
        
    n_cells = tf_adata.n_obs

    obs = tf_adata.obs.copy()
    n_tot = obs.condition.nunique()
    if exclude_pert_control:
        ctrl_mask = obs[pert_col] == ctrl_pert
        ctrl_conds = obs.loc[ctrl_mask, 'condition'].unique().tolist()
    else: 
        ctrl_conds = []
        
    all_conds = obs.condition.unique().tolist()
    all_cats, all_perts = map(np.array, zip(*[c.split('^') for c in all_conds]))
    cond_to_cat = dict(zip(all_conds, all_cats))
    cond_to_pert = dict(zip(all_conds, all_perts))
    
    perts_all = np.array([cond_to_pert[c] for c in all_conds])
    cats_all  = np.array([cond_to_cat[c]  for c in all_conds])

    uniq_perts = np.unique(perts_all)
    uniq_cats  = np.unique(cats_all)
    
    tot_per_pert = {p: np.sum(perts_all == p) for p in uniq_perts}
    tot_per_cat  = {c: np.sum(cats_all  == c) for c in uniq_cats}

    # exclude from split
    obs_noctrl = obs.loc[~obs.condition.isin(ctrl_conds)].copy()
    n_ctrl = len(ctrl_conds)
    train_frac_adj = ((train_frac * n_tot) - n_ctrl) / obs_noctrl.condition.nunique()
    test_frac_adj = 1 - train_frac_adj
    

    n_iter = 0
    conditions_unmet = True
    while conditions_unmet and (n_iter < max_attempts):
        train_conds, test_conds = train_test_split(obs_noctrl.condition.unique().tolist(), 
                                                   test_size = test_frac_adj, 
                                                   random_state = seed + n_iter, shuffle = True)
        train_conds += ctrl_conds


        # cell split matches condition split
        train_mask = obs.condition.isin(train_conds)
        n_train_cells = np.sum(train_mask)
        cell_train_frac = n_train_cells / n_cells
        deviation = np.abs(cell_train_frac - train_frac)
        outside_deviation = deviation > deviation_thresh


        # make sure it's not zero shot
        test_cats, test_perts = map(set, zip(*[cond.split('^') for cond in test_conds]))
        train_cats, train_perts = map(set, zip(*[cond.split('^') for cond in train_conds]))
        zero_shot = bool(test_cats - train_cats or test_perts - train_perts)

        train_perts_arr = np.array([cond_to_pert[c] for c in train_conds])
        train_cats_arr  = np.array([cond_to_cat[c]  for c in train_conds])

        under_pert_frac = False
        for p in uniq_perts:
            n_train_p = np.sum(train_perts_arr == p)
            frac = n_train_p / tot_per_pert[p]
            if frac < min_pert_frac:
                under_pert_frac = True
                break

        under_cat_frac = False
        for c in uniq_cats:
            n_train_c = np.sum(train_cats_arr == c)
            frac = n_train_c / tot_per_cat[c]
            if frac < min_cat_frac:
                under_cat_frac = True
                break

        conditions_unmet = outside_deviation or zero_shot or under_pert_frac or under_cat_frac
        n_iter += 1

    if n_iter >= max_attempts:
        return None
    else:
        split_dict = {
            'train_conds': train_conds, 
            'test_conds': test_conds, 
            'train_barcodes': obs.loc[train_mask, :].index.tolist(), 
            'test_barcodes': obs.loc[~train_mask, :].index.tolist()
        }
        return split_dict
    
# usage
# n_folds = 5
# split_dict = ood_split(
#     tf_adata = tf_adata, 
#     train_frac = (n_folds - 1)/n_folds,
#     min_pert_frac = 0.6, 
#     min_cat_frac = 0.5, 
#     deviation_thresh = 0.025, 
#     max_attempts = 1000, 
#     pert_col = 'ligand', 
#     ctrl_pert = 'CTRL', 
#     exclude_pert_control = True
    
# )
    
    
    
    
def ood_kfold_split(tf_adata,
                    n_folds: int,
                    min_pert_frac: float = 0.4,
                    min_cat_frac: float = 0.5,
                    deviation_thresh: float = 0.025,
                    max_attempts: int = 1000,
                    exclude_pert_control: bool = True,
                    pert_col: str = 'ligand',
                    ctrl_pert: str ='CTRL',
                    seed: int = 888):
    """
    K-fold CV version of ood_split.

    If exclude_pert_control=True:
      - folds are created ONLY over non-control conditions
      - control conditions are always appended to train_conds for every fold
      - test_conds never include controls

    Returns
    -------
    A list/dict of folds in the same format as ood_split,
    or None if no valid k-fold assignment is found within max_attempts.
    """

    n_cells = tf_adata.n_obs

    obs = tf_adata.obs.copy()
    n_tot = obs.condition.nunique()
    if exclude_pert_control:
        ctrl_mask = obs[pert_col] == ctrl_pert
        ctrl_conds = obs.loc[ctrl_mask, 'condition'].unique().tolist()
    else:
        ctrl_conds = []

    all_conds = obs.condition.unique().tolist()
    all_cats, all_perts = map(np.array, zip(*[c.split('^') for c in all_conds]))
    cond_to_cat = dict(zip(all_conds, all_cats))
    cond_to_pert = dict(zip(all_conds, all_perts))

    perts_all = np.array([cond_to_pert[c] for c in all_conds])
    cats_all  = np.array([cond_to_cat[c]  for c in all_conds])

    uniq_perts = np.unique(perts_all)
    uniq_cats  = np.unique(cats_all)

    tot_per_pert = {p: np.sum(perts_all == p) for p in uniq_perts}
    tot_per_cat  = {c: np.sum(cats_all  == c) for c in uniq_cats}

    # exclude from fold assignment (but will be appended to train in each fold if exclude_pert_control=True)
    obs_noctrl = obs.loc[~obs.condition.isin(ctrl_conds)].copy()
    noctrl_conds = obs_noctrl.condition.unique().tolist()

#     if exclude_pert_control:
#         ctrl_mask_cells = obs[pert_col] == ctrl_pert
#         n_ctrl_cells = np.sum(ctrl_mask_cells)
#         n_noctrl_cells = n_cells - n_ctrl_cells

#         train_frac_target = 1 - (1 / n_folds) * (n_noctrl_cells / n_cells)
#     else:
    train_frac_target = (n_folds - 1) / n_folds

    n_iter = 0
    while n_iter < max_attempts:

        rng = np.random.default_rng(seed + n_iter)
        rng.shuffle(noctrl_conds)

        folds = np.array_split(noctrl_conds, n_folds)
        
        
#         # build a map of condition -> #cells
#         cond_counts = obs_noctrl.condition.value_counts().to_dict()

#         # shuffle conditions to randomize ties / ordering, then sort by size desc
#         conds = noctrl_conds.copy()
#         rng = np.random.default_rng(seed + n_iter)
#         rng.shuffle(conds)
#         conds = sorted(conds, key=lambda c: cond_counts.get(c, 0), reverse=True)

#         # greedy bin packing into k folds (by cell counts)
#         fold_lists = [[] for _ in range(n_folds)]
#         fold_cell_sums = [0 for _ in range(n_folds)]

#         for cond in conds:
#             j = int(np.argmin(fold_cell_sums))
#             fold_lists[j].append(cond)
#             fold_cell_sums[j] += cond_counts.get(cond, 0)

#         folds = [np.array(f) for f in fold_lists]


        all_splits = {}
        all_valid = True

        for fold_idx in range(n_folds):

            test_conds = folds[fold_idx].tolist()
            train_conds = [
                cond
                for i, f in enumerate(folds)
                if i != fold_idx
                for cond in f.tolist()
            ]
            train_conds += ctrl_conds

            # cell split matches condition split (target is CV's train_frac_target)
            train_mask = obs.condition.isin(train_conds)
            test_mask  = obs.condition.isin(test_conds)

            n_train_cells = np.sum(train_mask)
            n_test_cells  = np.sum(test_mask)

            cell_train_frac = n_train_cells / (n_train_cells + n_test_cells)
            deviation = np.abs(cell_train_frac - train_frac_target)
            outside_deviation = deviation > deviation_thresh
            if outside_deviation:
                all_valid = False
                break  # move on to next attempt immediately

            # make sure it's not zero shot
            test_cats, test_perts = map(set, zip(*[cond.split('^') for cond in test_conds]))
            train_cats, train_perts = map(set, zip(*[cond.split('^') for cond in train_conds]))
            zero_shot = bool(test_cats - train_cats or test_perts - train_perts)
            if zero_shot:
                all_valid = False
                break  # move on to next attempt immediately

            train_perts_arr = np.array([cond_to_pert[c] for c in train_conds])
            train_cats_arr  = np.array([cond_to_cat[c]  for c in train_conds])

            under_pert_frac = False
            for p in uniq_perts:
                n_train_p = np.sum(train_perts_arr == p)
                frac = n_train_p / tot_per_pert[p]
                if frac < min_pert_frac:
                    under_pert_frac = True
                    break
            if under_pert_frac:
                all_valid = False
                break  # move on to next attempt immediately

            under_cat_frac = False
            for c in uniq_cats:
                n_train_c = np.sum(train_cats_arr == c)
                frac = n_train_c / tot_per_cat[c]
                if frac < min_cat_frac:
                    under_cat_frac = True
                    break
            if under_cat_frac:
                all_valid = False
                break  # move on to next attempt immediately

            all_splits[fold_idx] = {
                'train_conds': train_conds,
                'test_conds': test_conds,
                'train_barcodes': obs.loc[train_mask, :].index.tolist(),
                'test_barcodes': obs.loc[test_mask, :].index.tolist()
            }

        if all_valid and len(all_splits) == n_folds:
            return all_splits

        n_iter += 1

    return None


5-fold CV split that meets the following:
- Conditions are exactly split 80-20, # of cells falls within a 10 percentage poitns tolerance (70-90%); this is lenient, but splitting wouldn't converge otherwise
- Train contain all cats and perts in test, but not the combination (OOD)

We also ensure sufficient coverage of each cell type / perturbation by enforcing the following:
- Training conditions include at least 50% of all conditions associated with each perturbation 
- Training conditions include at least 60% of all conditions associated with each cell type

Note, true k-fold CV is only implemented on non control conditions, as we enforce that control conditions are always in the training split. 

In [5]:
all_splits = ood_kfold_split(
    tf_adata,
    n_folds = 5, 
    min_pert_frac = 0.5, 
    min_cat_frac = 0.6, 
    deviation_thresh = 0.1, 
    max_attempts = 5000, 
    exclude_pert_control = True, 
    pert_col = 'ligand', 
    ctrl_pert = 'CTRL', 
    seed = seed
)

# basic sanity checks
for k, split in all_splits.items():
    assert len(split['train_barcodes']) + len(split['test_barcodes']) == tf_adata.shape[0]
    assert np.abs(len(split['train_barcodes']) / tf_adata.shape[0] - 0.8) < 0.1
    assert 'CTRL' not in {cond.split('^')[1] for cond in split['test_conds']}
    assert [cond.split('^')[1] for cond in split['train_conds']].count('CTRL') == 4
    
with open(os.path.join(data_path, 'processed', author + '_5foldCV_splits.json'), "w") as f:
    json.dump(all_splits, f)
    